## Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import re

## Importing data

Columns to drop

In [ ]:
columns_to_not_use = [
# Same values for all rows
    'record_type' ,
    'record_subclass' ,
    'confirmation_location' ,
    'dia_bp' ,
    'group_number' ,
    'manufacturer_code' ,
    'patient_age_units' ,
    'patient_height' ,
    'patient_weight' ,
    'reason_for_procedure' ,
    'sys_bp' ,
    'text_size' ,
    'pp_interval' ,

# No value at all
    'assisting_provider' ,
    'attending_phy' ,
    'patient_middle_name' ,
    'patient_secondary_id' ,
    'patient_ssn' ,
    'procedural_diagnosis' ,
    'referral_source' ,
    'referring_phy' ,
]

Columns to format in a particular type

In [ ]:
sure_string_cols = [
    'pseudopaz',
    'patient_date_of_birth', # Because of the inconsistency ''dd.mm.yyyy' vs 'mm/dd/yyyy'
]
dtype = {col: 'string' for col in sure_string_cols}

string_cat_cols = [
    'site_number', # ???
    'record_status',
    'medications',
    'patient_height_units' ,
    'patient_location' ,
    'patient_race' ,
    'patient_race_str' ,
    'patient_room' ,
    'patient_sex' ,
    'patient_weight_units' ,
    'transmit_device_number' ,
    'reason_for_change' ,
]
dtype.update({col: 'string' for col in string_cat_cols})

Import the excel file

In [ ]:
df = pd.read_excel('../raw_data/dati_escribe.xlsx',
    usecols=lambda col: col not in columns_to_not_use,
    dtype_backend='numpy_nullable', # For allowing the `string` dtype
    dtype=dtype
)

# Make the index start from 2 (as the original excel)
df.index = df.index + 2

## Parsing and reformatting

Defining `format_phy_name` function

In [ ]:
def format_phy_name(value):
    '''
    Requires pandas.
    TODO
    '''
    # Handle NA values
    if pd.isna(value):
        return pd.NA

    # Count whitespaces
    dot_count = value.count('.')

    # Handle the only name with one dot
    if dot_count == 1:
        return value

    elif dot_count == 2:
        second_dot = value.rfind('.')
        name = value[second_dot -1 : second_dot]
        surname = value[value.rfind('.') +1 : ].strip()
        formatted_name = 'DR-' + name + '_' + surname
        
        return formatted_name

### Formatting `overreading_phy` column

In [ ]:
# Only name with one dot
df['overreading_phy'] = df['overreading_phy'].replace('DR. GP BROSO', 'DR-GP_BROSO')

# Run the function
df['overreading_phy'] = df['overreading_phy'].apply(format_phy_name).astype('string')

### Splitting and formatting `interp_text` column

In [ ]:
def split_column(value):

    # If row is already a NA value, leave unchanged
    if pd.isna(value):
        return pd.Series([pd.NA, pd.NA, pd.NA])

    # Split on the first ~~ from right
    parts = value.rsplit('~~', maxsplit=1)

    # Place a NA in the missing splitted parts
    for i, part in enumerate(parts):
        if part == '':
            parts[i] = pd.NA

    # Parse 'parts[1]'
    if parts[1].startswith('REPORT NON CONFERM'):
        doc, datetime = 'RNC', pd.NaT
    elif parts[1].startswith('Refertato da _________________'):
        doc, datetime = 'phy_NA', pd.NaT
    elif parts[1].startswith('Refertato da '):
        doc_pro, date, hour = (re.sub(' +', ' ', parts[1])).strip().rsplit(' ', maxsplit=2)
        doc = format_phy_name(doc_pro[13:])
        datetime = pd.to_datetime('{} {}'.format(date, hour), format='%d/%m/%Y %H.%M.%S')

    if len(parts) == 2:
        return pd.Series([parts[0], doc, datetime])
    else:
        raise Exception('Abnormal row')

Pre-format row 1685, the only problematic

In [ ]:
original = df.at[1685, 'interp_text']
df.at[1685, 'interp_text'] = '~' + original

In [ ]:
# Get position and split
original_col_name = 'interp_text'
original_position = df.columns.get_loc(original_col_name)

# Split the column
split_df = df[original_col_name].apply(split_column)
split_df.columns = ['diagnosis', 'phy', 'phy_date']
split_df[['diagnosis', 'phy']] = split_df[['diagnosis', 'phy']].astype('string')

# Reconstruct dataframe: columns before + new columns + columns after
df = pd.concat([
    df.iloc[:, :original_position
        # +1 # Decomment for keeping the original column
    ],
    split_df,
    df.iloc[:, original_position+1:]
], axis=1)

### Merging similar values in columns

#### `comment_field`

Values are `{..., 'Dolore toracico': 8,'Dolore torace': 2, ...}` - `Dolore torace` values are merged to `Dolore toracico`

In [ ]:
df['comment_field'] = df['comment_field'].replace('Dolore torace', 'Dolore toracico')

#### `note1`

Merging:
- `DOL. TORACICO` and `DOL TOR` and `DOLORETORAC` and `DOLORE TORACICO`
    - to `TORACALGIA`
- `ADDOMINALG` and `DOLORE ADDOMIN`
    - to `ADDOMINALGIA`
- `Dispnea`
    - to `DISPNEA`

In [ ]:
df['note1'] = df['note1'].replace(['DOLORE TORACICO', 'DOLORETORAC', 'DOL. TORACICO', 'DOL TOR'], 'TORACALGIA')
df['note1'] = df['note1'].replace(['ADDOMINALG', 'DOLORE ADDOMIN'], 'ADDOMINALGIA')
df['note1'] = df['note1'].replace(['Dispnea'], 'DISPNEA')

#### `note2` (tanti auguri)

Merging:
- `.` and `.`
    - to `.`

In [ ]:
# df[''] = df[''].replace([''], '')

#### `patient_location`

Merging:
- `OTO`
    - to `Oto`

In [ ]:
df['patient_location'] = df['patient_location'].replace(['OTO'], 'Oto')

#### ``

Merging:
- `.` and `.`
    - to `.`

In [ ]:
# df[''] = df[''].replace([''], '')

### Reformatting `patient_date_of_birth` column

#### Proof of consistency between american and european formatting

Most of the dates are written with `.` as separator, only 33 of them with `/`:

In [ ]:
dot_rows = df['patient_date_of_birth'].str.contains(r'\.', na=False).sum()
slash_rows = df['patient_date_of_birth'].str.contains('/', na=False).sum()
print('Number of rows formatted with dot:', dot_rows)
print('Number of rows formatted with slash:', slash_rows)

Hypotesis: _dates written with dots as separator are written in the european way (`dd.mm.yyyy`), dates with slash are written in the american way (`mm/dd/yyyy`)_.

There can't be an absolute proof of this, because some day numbers are lower or equal than 12.

But:
* if, for slash dates, the first field never exceeds 12 but the second often does
* if, for dot dates, the first field often exceeds 12 while the second never does

the probability that the hypotesis is true is very high. So let's test it.

In [ ]:
# Remove surrounding spaces
df['patient_date_of_birth'] = df['patient_date_of_birth'].str.strip()

# Separate rows by separator
slash_mask = df['patient_date_of_birth'].str.contains('/', na=False)
dot_mask = df['patient_date_of_birth'].str.contains(r'\.', na=False)

# Filter values from patient_date_of_birth column only for rows where slash_mask is True.
slash_dates = df.loc[slash_mask, 'patient_date_of_birth']
dot_dates = df.loc[dot_mask, 'patient_date_of_birth']

# Split the dates components and rename the columns for clarity
slash_parts = slash_dates.str.split('/', expand=True).astype(int)
slash_parts.columns = ['s_part1', 's_part2', 's_year']
dot_parts = dot_dates.str.split(r'\.', expand=True).astype(int)
dot_parts.columns = ['d_part1', 'd_part2', 'd_year']

Proof for slash-formatted days

In [ ]:
slash_first_gt12 = (slash_parts['s_part1'] > 12).sum()
slash_second_gt12 = (slash_parts['s_part2'] > 12).sum()

print('SLASH DATES')
print(f'Total slash dates: {len(slash_parts)}')
print(f'First field > 12 : {slash_first_gt12}')
print(f'Second field > 12: {slash_second_gt12}')

# Rows that prove mm/dd/yyyy
slash_proof = slash_dates[slash_parts['s_part2'] > 12]
slash_proof.shape

Proof for dot-formatted days

In [ ]:
dot_first_gt12 = (dot_parts['d_part1'] > 12).sum()
dot_second_gt12 = (dot_parts['d_part2'] > 12).sum()

print('DOT DATES')
print(f'Total dot dates: {len(dot_parts)}')
print(f'First field > 12 : {dot_first_gt12}')
print(f'Second field > 12: {dot_second_gt12}')

# Rows that prove dd.mm.yyyy
dot_proof = dot_dates[dot_parts['d_part1'] > 12]

So in a total of 1787 dates:
* 16 dates out of 33 suggest that slash-formatted are written in the format `mm/dd/yyyy`
* 1001 dates out of 1754 suggest that dot-formatted are written in the format `mm/dd/yyyy`
* no date goes against the hypotesis

#### Reformatting and converting to datetime type

In [ ]:
# Clean eventual whitespaces
df['patient_date_of_birth'] = df['patient_date_of_birth'].str.strip()

# Make masks for the two separators
slash_mask = df['patient_date_of_birth'].str.contains('/', na=False)
dot_mask = df['patient_date_of_birth'].str.contains(r'\.', na=False)

# Temporary Series to hold parsed dates
parsed_dates = pd.Series(pd.NaT, index=df.index)

# Parse slash dates (mm/dd/yyyy)
parsed_dates.loc[slash_mask] = pd.to_datetime(
    df.loc[slash_mask, 'patient_date_of_birth'],
    format='%m/%d/%Y',
    errors='coerce'
)

# Parse dot dates (dd.mm.yyyy)
parsed_dates.loc[dot_mask] = pd.to_datetime(
    df.loc[dot_mask, 'patient_date_of_birth'],
    format='%d.%m.%Y',
    errors='coerce'
)

# Replace original column
df['patient_date_of_birth'] = parsed_dates

# Check if some rows have failed the parsing
failed_rows = df[df['patient_date_of_birth'].isna()]
print(f'Failed rows: {len(failed_rows)}')

# # Rename column to a different name
# df = df.rename(columns={
#     'patient_date_of_birth': 'normalized_date'
# })

## Final checks

In [ ]:
df

In [ ]:
df.info()

## Export

In [ ]:
# # Parquet
# df.to_parquet('../raw_data/parquets/df-1-escribe.parquet')

# # Pickle
# df.to_pickle('../raw_data/pickles/df-1-escribe.pickle')

## Inconsistency between `pseudopaz` and other columns

#### Proof of the inconsistency

`patient_age`, `patient_date_of_birth`, `patient_race`, `patient_sex` are inconsistent

In [ ]:
# (df.groupby('pseudopaz')['patient_age'].nunique() == 1).all()
# (df.groupby('pseudopaz')['patient_date_of_birth'].nunique() == 1).all()
# (df.groupby('pseudopaz')['patient_race'].nunique() == 1).all()
# (df.groupby('pseudopaz')['patient_sex'].nunique() == 1).all()

#### See which `pseudopaz` have multiple characteristics that should be unique

In [ ]:
# incons = df.groupby('pseudopaz')['patient_date_of_birth'].nunique()
# incons = incons[incons > 1].sort_values(ascending=False)
# incons